# TI3145TU Final Assignment 
## Health Insurance 

We hope you enjoy this assignment, good luck!

Student names: XXX

Student numbers: XXX

### Imports

In [1]:
import numpy as np
import pandas as pd

from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import SGDRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

import matplotlib.pyplot as plt

### Load data

In [2]:
# These are your training samples along with their labels
data = pd.read_csv('health_insurance_train.csv')
data.head()

# You need to extract the features and the regression target. The regression target is 'whrswk'. 

,whrswk,hhi,whi,hhi2,education,race,hispanic,experience,kidslt6,kids618,husby,region
0,40.0,no,yes,yes,13-15years,white,no,17.0,0.0,1.0,22.000,south
1,40.0,no,yes,yes,13-15years,white,no,4.0,1.0,0.0,15.000,south
2,0.0,yes,no,yes,16years,white,no,21.0,0.0,1.0,99.999,other
3,40.0,no,no,yes,13-15years,white,no,22.0,NaN,NaN,60.000,northcentral
4,35.0,no,yes,no,12years,white,no,15.0,0.0,2.0,0.000,south


### Pipeline 1

In [3]:
import numpy as np
import pandas as pd
import sklearn

from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import SGDRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
import matplotlib.pyplot as plt

# These are your training samples along with their labels
data = pd.read_csv('health_insurance_train.csv')
# Here we separate target (y) and features (X)
X = data.drop('whrswk', axis=1)
y = data['whrswk']
# divide columns into numerical and non-numerical features
#numerical_feats = ['experience', 'kidslt6', 'kids618', 'husby']
#categorical_feats = ['hhi', 'whi', 'hhi2', 'education', 'race', 'hispanic', 'region']
# transformer for numbers
#numerical_transf = Pipeline(steps=[('imputer', SimpleImputer(strategy="median")), ('scaler', StandardScaler())])
# transformer for non-numbers
#categorical_transf = Pipeline(steps=[('imputer', SimpleImputer(strategy="most_frequent")), ('encoder', OneHotEncoder())])
#
#preprocessor = ColumnTransformer(
#    transformers=[
#        ('num', numerical_transf, numerical_feats),
#        ('cat', categorical_transf, categorical_feats)
#    ]
#)

#preprocessor.fit(X)

#X_transformed = preprocessor.transform(X)

#print(X_transformed[0])
# You need to extract the features and the regression target. The regression target is 'whrswk'.


# ---------------------
# Pipeline 1: Simple baseline pipeline
# ---------------------
numerical_feats_1 = ['experience', 'kidslt6', 'kids618', 'husby']
categorical_feats_1 = ['hhi', 'whi', 'hhi2', 'education', 'race', 'hispanic', 'region']

numerical_transf_1 = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transf_1 = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor1 = ColumnTransformer([
    ('num', numerical_transf_1, numerical_feats_1),
    ('cat', categorical_transf_1, categorical_feats_1)
])

# ---------------------
# Pipeline 2: Engineered + scaled pipeline
# ---------------------
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor

# Separate numerical features — we'll split them into two groups
poly_feat = ['experience']
other_num_feats = ['kidslt6', 'kids618', 'husby']
cat_feats = ['hhi', 'whi', 'hhi2', 'education', 'race', 'hispanic', 'region']

# Transformer for the polynomial 'experience' feature
poly_transf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler())
])

# Transformer for other numeric features (no polynomial)
other_num_transf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Transformer for categorical features
cat_transf = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine all transformations
preprocessor2 = ColumnTransformer([
    ('poly', poly_transf, poly_feat),
    ('num', other_num_transf, other_num_feats),
    ('cat', cat_transf, cat_feats)
])



### Dummy regressor

In [4]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

dummy = DummyRegressor(strategy="mean")
dummy.fit(X, y)
y_pred = dummy.predict(X)
mae_baseline = mean_absolute_error(y, y_pred)
print("Baseline MAE (DummyRegressor, mean strategy):", mae_baseline)

Baseline MAE (DummyRegressor, mean strategy): 16.97666512


### Model training

In [5]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, mean_absolute_error


models = {
    "KNN": KNeighborsRegressor(),
    "SGD": SGDRegressor(random_state=42),  # add random_state for reproducibility
    "RandomForest": RandomForestRegressor(random_state=42),
    "DecisionTree": DecisionTreeRegressor(random_state=42)
}
results = {}

# Loop over models
for name, model in models.items():
    # Create full pipeline: preprocessing + model
    full_pipeline = Pipeline([('preprocessor', preprocessor2),  ('regressor', model)])
    #full_pipeline = make_pipeline2_simple(model)
    # Cross-validated MAE (5-fold CV)
    cv_scores = cross_val_score(full_pipeline, X, y, cv=5, scoring='neg_mean_absolute_error')
    
    # Convert negative MAE to positive
    mae = -cv_scores.mean()
    
    results[name] = mae
    print(name, results[name])

    


KNN 12.256519999999998
SGD 11.980481541984751
RandomForest 11.8080129018426
DecisionTree 14.157187619047619


### Tuning

In [ ]:
# =======================================================
# 3.2 Nested Cross-Validation with GridSearchCV
# =======================================================
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import SGDRegressor

# =======================================================
# Outer and inner CV
# =======================================================
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)  # Outer loop for evaluation
inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)  # Inner loop for tuning

# =======================================================
# Parameter grids (with max_iter for SGD)
# =======================================================
param_grids = {
    'KNN': {
        'regressor__n_neighbors': [9, 11, 13, 15, 17, 19],
        'regressor__weights': ['uniform', 'distance'],
        'regressor__p': [1, 2]
    },
    'SGD': {
        'regressor__alpha': [1e-5, 1e-4, 1e-1],
        'regressor__penalty': ['l2', 'l1', 'elasticnet'],
        'regressor__learning_rate': ['constant', 'optimal'],
        'regressor__eta0': [0.0001, 0.001, 0.01],
        'regressor__max_iter': [100, 200, 500]
    },
    'DecisionTree': {
        'regressor__max_depth': [5, 6, 7, None],
        'regressor__min_samples_split': [2, 5, 10, 15],
        'regressor__min_samples_leaf': [1, 2, 4],
        'regressor__criterion': ['squared_error', 'friedman_mse']
    },
    'RandomForest': {
        'regressor__n_estimators': [200, 300, 400],
        'regressor__max_depth': [5, 10, 12, None],
        'regressor__min_samples_split': [2, 5, 10, 13],
        'regressor__min_samples_leaf': [1, 2, 4, 6]
    }
}

# =======================================================
# Storage for results
# =======================================================
nested_results = {}

# =======================================================
# Loop over models
# =======================================================
for name, model in models.items():
    print(f"\n🔍 Nested CV for {name}")

    # Build pipeline
    pipeline = Pipeline([
        ('preprocessor', preprocessor2),
        ('regressor', model)
    ])

    # Inner GridSearchCV
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[name],
        scoring='neg_mean_absolute_error',
        cv=inner_cv,
        n_jobs=-1,
        verbose=0
    )

    # Store best params for each outer fold
    outer_best_params = []
    outer_mae_scores = []

    # Manual outer CV loop to capture best params per fold
    for train_idx, test_idx in outer_cv.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Fit inner GridSearch on training fold
        grid_search.fit(X_train, y_train)

        # Get best estimator and evaluate on test fold
        best_model = grid_search.best_estimator_
        y_pred = best_model.predict(X_test)
        mae = np.mean(np.abs(y_test - y_pred))

        # Store fold results
        outer_best_params.append(grid_search.best_params_)
        outer_mae_scores.append(mae)

        print(f"  🔸 Outer fold MAE: {mae:.4f}")
        print(f"     Best params: {grid_search.best_params_}")

    # Compute overall stats
    mean_mae = np.mean(outer_mae_scores)
    std_mae = np.std(outer_mae_scores)

    nested_results[name] = {
        'Mean_MAE': mean_mae,
        'Std_MAE': std_mae,
        'Best_Params_Per_Fold': outer_best_params
    }

    print(f"\n✅ Nested CV Result for {name}: {mean_mae:.4f} ± {std_mae:.4f}")
    print("📋 Best params per outer fold:")
    for i, p in enumerate(outer_best_params, 1):
        print(f"   Fold {i}: {p}")

# =======================================================
# Summarize results
# =======================================================
nested_summary = pd.DataFrame({
    k: {'Mean_MAE': v['Mean_MAE'], 'Std_MAE': v['Std_MAE']} for k, v in nested_results.items()
}).T

print("\n=== 🧾 Nested CV Summary ===")
display(nested_summary)

# Optional: visualize results
nested_summary['Mean_MAE'].plot(
    kind='bar',
    title='Nested CV Mean MAE per Model',
    ylabel='Mean Absolute Error',
    rot=0,
    yerr=nested_summary['Std_MAE']
)
plt.tight_layout()
plt.show()

# =======================================================
# SGD Training Curve Visualization
# =======================================================
best_sgd_params = nested_results['SGD']['Best_Params_Per_Fold'][0]  # Use first fold's best params

# Clean params for SGDRegressor
sgd_params_clean = {k.replace('regressor__', ''): v for k, v in best_sgd_params.items()}
sgd_params_clean['warm_start'] = True
sgd_params_clean['verbose'] = 0

# Initialize SGD model
sgd_model = SGDRegressor(**sgd_params_clean)

# Fit SGD on the full dataset
X_processed = preprocessor2.fit_transform(X)
sgd_model.fit(X_processed, y)

# Plot the training curve
if hasattr(sgd_model, 'loss_curve_'):
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(sgd_model.loss_curve_)+1), sgd_model.loss_curve_, marker='o')
    plt.title("SGDRegressor Training Curve (Loss per Iteration)")
    plt.xlabel("Iteration")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.show()
else:
    print("Loss curve not available. Make sure SGDRegressor was trained with max_iter > 1.")



🔍 Nested CV for KNN


/usr/local/lib/python3.10/site-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


  🔸 Outer fold MAE: 11.9392
     Best params: {'regressor__n_neighbors': 13, 'regressor__p': 1, 'regressor__weights': 'uniform'}
  🔸 Outer fold MAE: 11.4743
     Best params: {'regressor__n_neighbors': 13, 'regressor__p': 1, 'regressor__weights': 'uniform'}
  🔸 Outer fold MAE: 12.5854
     Best params: {'regressor__n_neighbors': 17, 'regressor__p': 1, 'regressor__weights': 'distance'}
  🔸 Outer fold MAE: 11.8031
     Best params: {'regressor__n_neighbors': 15, 'regressor__p': 1, 'regressor__weights': 'uniform'}
  🔸 Outer fold MAE: 11.6419
     Best params: {'regressor__n_neighbors': 15, 'regressor__p': 1, 'regressor__weights': 'uniform'}

✅ Nested CV Result for KNN: 11.8888 ± 0.3816
📋 Best params per outer fold:
   Fold 1: {'regressor__n_neighbors': 13, 'regressor__p': 1, 'regressor__weights': 'uniform'}
   Fold 2: {'regressor__n_neighbors': 13, 'regressor__p': 1, 'regressor__weights': 'uniform'}
   Fold 3: {'regressor__n_neighbors': 17, 'regressor__p': 1, 'regressor__weights': 'distan

/usr/local/lib/python3.10/site-packages/sklearn/linear_model/_stochastic_gradient.py:1616: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/usr/local/lib/python3.10/site-packages/sklearn/linear_model/_stochastic_gradient.py:1616: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/usr/local/lib/python3.10/site-packages/sklearn/linear_model/_stochastic_gradient.py:1616: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/usr/local/lib/python3.10/site-packages/sklearn/linear_model/_stochastic_gradient.py:1616: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/usr/local/lib/python3.10/site-packages/sklearn/linear_model/_stocha

  🔸 Outer fold MAE: 12.2452
     Best params: {'regressor__alpha': 1e-05, 'regressor__eta0': 0.001, 'regressor__learning_rate': 'constant', 'regressor__max_iter': 100, 'regressor__penalty': 'l1'}


...

  🔸 Outer fold MAE: 12.5726
     Best params: {'regressor__alpha': 1e-05, 'regressor__eta0': 0.001, 'regressor__learning_rate': 'constant', 'regressor__max_iter': 100, 'regressor__penalty': 'l1'}


/usr/local/lib/python3.10/site-packages/sklearn/linear_model/_stochastic_gradient.py:1616: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/usr/local/lib/python3.10/site-packages/sklearn/linear_model/_stochastic_gradient.py:1616: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/usr/local/lib/python3.10/site-packages/sklearn/linear_model/_stochastic_gradient.py:1616: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/usr/local/lib/python3.10/site-packages/sklearn/linear_model/_stochastic_gradient.py:1616: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/usr/local/lib/python3.10/site-packages/sklearn/linear_model/_stocha

  🔸 Outer fold MAE: 11.8278
     Best params: {'regressor__alpha': 1e-05, 'regressor__eta0': 0.0001, 'regressor__learning_rate': 'constant', 'regressor__max_iter': 200, 'regressor__penalty': 'l1'}


/usr/local/lib/python3.10/site-packages/sklearn/linear_model/_stochastic_gradient.py:1616: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/usr/local/lib/python3.10/site-packages/sklearn/linear_model/_stochastic_gradient.py:1616: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/usr/local/lib/python3.10/site-packages/sklearn/linear_model/_stochastic_gradient.py:1616: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/usr/local/lib/python3.10/site-packages/sklearn/linear_model/_stochastic_gradient.py:1616: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/usr/local/lib/python3.10/site-packages/sklearn/linear_model/_stocha

  🔸 Outer fold MAE: 12.0208
     Best params: {'regressor__alpha': 1e-05, 'regressor__eta0': 0.001, 'regressor__learning_rate': 'constant', 'regressor__max_iter': 100, 'regressor__penalty': 'l1'}

✅ Nested CV Result for SGD: 11.9941 ± 0.4248
📋 Best params per outer fold:
   Fold 1: {'regressor__alpha': 1e-05, 'regressor__eta0': 0.001, 'regressor__learning_rate': 'constant', 'regressor__max_iter': 100, 'regressor__penalty': 'l1'}
   Fold 2: {'regressor__alpha': 1e-05, 'regressor__eta0': 0.001, 'regressor__learning_rate': 'constant', 'regressor__max_iter': 100, 'regressor__penalty': 'l1'}
   Fold 3: {'regressor__alpha': 1e-05, 'regressor__eta0': 0.001, 'regressor__learning_rate': 'constant', 'regressor__max_iter': 100, 'regressor__penalty': 'l1'}
   Fold 4: {'regressor__alpha': 1e-05, 'regressor__eta0': 0.0001, 'regressor__learning_rate': 'constant', 'regressor__max_iter': 200, 'regressor__penalty': 'l1'}
   Fold 5: {'regressor__alpha': 1e-05, 'regressor__eta0': 0.001, 'regressor__learni

### Autograder 

In the autograder you will need to provide two things: 1) estimate of the MAE of your model on unseen data, 2) the predictions on the autograder data. For the autograder data we only provide the features and not the regression targets. Thus, you cannot compute the MAE on this data yourself - you need to estimate that with the data provided above. 

In [ ]:
data_autograder = pd.read_csv('health_insurance_autograde.csv')
data_autograder.head()


In [ ]:
# TODO Replace this with your own estimate of the MAE of your best model
estimate_MAE_on_new_data = 11.816

# TODO Replace this with the predictions of your best model
# via e.g. prediction = model.predict(data_autograder)
predictions_autograder_data = np.array([-1] * 17272)

# Upload this file to the Vocareum autograder:
result = np.append(estimate_MAE_on_new_data, predictions_autograder_data)
pd.DataFrame(result).to_csv("autograder_submission.txt", index=False, header=False)